In [1]:
HANOI_BBOX = {
    "lon_min": 105.3,
    "lon_max": 106.0,
    "lat_min": 20.5,
    "lat_max": 21.4
}


In [3]:
import pandas as pd
df = pd.read_csv("openaq_vietnam_hourly.csv")
print(df.shape)           # xem số rows/cols
print(df.dtypes)          # xem kiểu dữ liệu
print(df.isnull().sum())  # xem missing values
print(df["parameter"].value_counts())   # các chỉ số có gì
print(df["city"].value_counts().head(20))  # phân bố theo tỉnh

FileNotFoundError: [Errno 2] No such file or directory: 'openaq_vietnam_hourly.csv'

In [ ]:
hanoi_names = ["Hanoi", "Ha Noi", "Hà Nội", "hanoi"]
df_hn = df[df["city"].str.lower().isin([x.lower() for x in hanoi_names])]
df_hn = df[
    (df["latitude"]  >= 20.5) & (df["latitude"]  <= 21.4) &
    (df["longitude"] >= 105.3) & (df["longitude"] <= 106.0)
]
df_hn = df[
    df["city"].str.lower().str.contains("hanoi|ha noi|hà nội", na=False) |
    (
        (df["latitude"]  >= 20.5) & (df["latitude"]  <= 21.4) &
        (df["longitude"] >= 105.3) & (df["longitude"] <= 106.0)
    )
]
print(f"Stations tại Hà Nội: {df_hn['location_id'].nunique()}")
print(f"Rows: {len(df_hn)}")

In [ ]:
# Parse datetime
df_hn["datetime_utc"] = pd.to_datetime(df_hn["datetime_utc"], utc=True)

# Loại bỏ giá trị không hợp lệ theo từng chỉ số
VALID_RANGES = {
    "pm25": (0, 500),    # µg/m³ — WHO: >500 là sensor lỗi
    "pm10": (0, 600),
    "no2":  (0, 200),    # µg/m³
    "co":   (0, 50),     # mg/m³
    "so2":  (0, 500),    # µg/m³
    "o3":   (0, 500),    # µg/m³
}

def clean_value(row):
    param = row["parameter"].lower()
    val   = row["value"]
    if param not in VALID_RANGES:
        return val
    lo, hi = VALID_RANGES[param]
    if val < lo or val > hi:
        return float("nan")
    return val

df_hn["value_clean"] = df_hn.apply(clean_value, axis=1)

# Loại rows có coverage_pct thấp (sensor không đo đủ trong giờ đó)
df_hn = df_hn[df_hn["coverage_pct"] >= 50]

print(f"Rows sau clean: {len(df_hn)}")
print(f"NaN sau clean: {df_hn['value_clean'].isna().sum()}")

In [ ]:
PARAMS = ["pm25", "no2", "co", "so2", "o3"]
df_filtered = df_hn[df_hn["parameter"].str.lower().isin(PARAMS)].copy()
df_filtered["parameter"] = df_filtered["parameter"].str.lower()

# Pivot
df_pivot = df_filtered.pivot_table(
    index=["location_id", "location_name", "latitude", "longitude", "datetime_utc"],
    columns="parameter",
    values="value_clean",
    aggfunc="mean"  
).reset_index()

df_pivot.columns.name = None  # bỏ tên index cột
print(df_pivot.shape)
print(df_pivot.head())

In [4]:
import json
import pandas as pd
import glob
from pathlib import Path

def parse_weather_file(filepath: str) -> pd.DataFrame:
    """Parse 1 file JSON của WeatherAPI thành DataFrame hourly."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    location = data.get("location", {})
    rows = []
    for day in data.get("forecast", {}).get("forecastday", []):
        for h in day.get("hour", []):
            row = {
                "province":     Path(filepath).parent.name,
                "lat":          location.get("lat"),
                "lon":          location.get("lon"),
                "tz_id":        location.get("tz_id"),
                "time":         h.get("time"),
                "time_epoch":   h.get("time_epoch"),
                "temp_c":       h.get("temp_c"),
                "humidity":     h.get("humidity"),
                "pressure_mb":  h.get("pressure_mb"),
                "precip_mm":    h.get("precip_mm"),
                "cloud":        h.get("cloud"),
                "wind_kph":     h.get("wind_kph"),
                "wind_degree":  h.get("wind_degree"),
                "wind_dir":     h.get("wind_dir"),
                "gust_kph":     h.get("gust_kph"),
                "vis_km":       h.get("vis_km"),
                "uv":           h.get("uv"),
                "condition_text": h.get("condition", {}).get("text"),
                "condition_code": h.get("condition", {}).get("code"),
                "is_day":       h.get("is_day"),
                "will_it_rain": h.get("will_it_rain"),
                "chance_of_rain": h.get("chance_of_rain"),
            }
            rows.append(row)
    return pd.DataFrame(rows)

# Load tất cả file JSON của Hà Nội
all_files = glob.glob(".data/weather/Hanoi/*.json") + glob.glob(".data/weather/Ha Noi/*.json")
print(f"Tìm thấy {len(all_files)} file JSON")

dfs = [parse_weather_file(f) for f in all_files]
df_weather = pd.concat(dfs, ignore_index=True)
print(f"Tổng rows: {len(df_weather)}")

Tìm thấy 0 file JSON


ValueError: No objects to concatenate